## 11b_cz_pl_model_run
  Expected radiation matrix for CZ obec and PL gmina with PyTDLM, saved with run time and peak
  memory. Output: data/interim/11b_cz_pl_model_run/, model_runs_summary.csv. Environment: nuts3_tdlm

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# --- Config (all changes are made here) ---
ROOT_MARKER = Path("data") / "interim"                       # a directory that exists only in the project root
LEVELS = {"CZ": "obec", "PL": "gmina"}
IN_DIR = Path("data") / "interim" / "09_lenormand_multilevel"
OUT_DIR = Path("data") / "interim" / "11b_cz_pl_model_run"
PROC_DIR = Path("data") / "processed" / "11b_cz_pl_model_run"
UNITS_FILE = "{country}_{level}_units.csv"
DIST_FILE = "{country}_{level}_distance_km.npy"
OBS_FILE = "{country}_{level}_od_observed.npy"
ID_COL, POP_COL = "unit_id", "population"
INPUT_CHECK_CSV = PROC_DIR / "input_check.csv"
SUM_RTOL = 1e-9

# find the project root and work from there (relative paths only)
root = Path.cwd()
while not (root / ROOT_MARKER).exists() and root != root.parent:
    root = root.parent
if not (root / ROOT_MARKER).exists():
    raise FileNotFoundError("A projekt gyökere nem található (data/interim hiányzik)")
os.chdir(root)
OUT_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)


def mem_available_gb():
    """Available memory in GB from /proc/meminfo (None if not readable)."""
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    return int(line.split()[1]) / 1024 ** 2
    except OSError:
        return None
    return None


check_rows = []
for country, level in LEVELS.items():
    fmt = dict(country=country, level=level)
    u = pd.read_csv(IN_DIR / UNITS_FILE.format(**fmt), dtype={ID_COL: str})
    n = len(u)
    pop = u[POP_COL].to_numpy(float)
    dist = np.load(IN_DIR / DIST_FILE.format(**fmt), mmap_mode="r")
    obs = np.load(IN_DIR / OBS_FILE.format(**fmt), mmap_mode="r")
    for name_, arr in (("distance", dist), ("observed", obs)):
        if arr.shape != (n, n):
            raise ValueError(f"{country}: a {name_} mátrix alakja {arr.shape}, az egységek száma {n}")
    if np.isnan(pop).any() or (pop < 0).any():
        raise ValueError(f"{country}: hiányzó vagy negatív népesség")
    diag_max = float(np.abs(np.diagonal(dist)).max())
    n_c = float(obs.sum(dtype=np.float64))                    # N_c = total of the observed matrix
    rate = n_c / pop.sum()
    out_trips = pop * rate
    if abs(out_trips.sum() / n_c - 1) > SUM_RTOL:
        raise ValueError(f"{country}: az out_trips összege nem egyezik az N_c értékkel")
    n_zero = int((pop == 0).sum())
    mem_low_gb = 2 * n * n * 8 / 1024 ** 3                    # at least: float64 distance copy + float64 output matrix
    avail = mem_available_gb()
    print(f"[{country} {level}] egység: {n} | N_c: {n_c:,.0f} | N: {pop.sum():,.0f} | N_c/N: {rate:.4f} | "
          f"nulla népességű egység: {n_zero}")
    print(f"    távolság: {dist.dtype}, átló legnagyobb értéke: {diag_max} | memóriaigény legalább: {mem_low_gb:.2f} GB | "
          f"elérhető memória: {'ismeretlen' if avail is None else f'{avail:.1f} GB'}")
    check_rows.append(dict(country=country, level=level, n_units=n, N_c=n_c, N=pop.sum(), rate=rate,
                           n_zero_pop=n_zero, diag_max_dist=diag_max, mem_lower_bound_gb=mem_low_gb))
    del u, pop, dist, obs

pd.DataFrame(check_rows).round(6).to_csv(INPUT_CHECK_CSV, index=False)
print(f"\nMentve: {INPUT_CHECK_CSV.as_posix()}")

In [ ]:
import time
import resource
from TDLM import tdlm

# --- Config (all changes are made here) ---
RUN_COUNTRIES = ["PL"]            # run one country at a time: first PL, then change to ["CZ"]
LAW, RAD_EXPONENT = "Rad", 1.0    # Rad has no fitted parameter
MAT_FILE = "{country}_Rad_UM_expected.npy"
SUMMARY_CSV = PROC_DIR / "model_runs_summary.csv"
SUM_RTOL_RUN = 1e-6
ROW_RTOL = 1e-6


def load_inputs(country, level):
    """Population, distance (memory-mapped), observed total N_c and out_trips of one country."""
    fmt = dict(country=country, level=level)
    u = pd.read_csv(IN_DIR / UNITS_FILE.format(**fmt), dtype={ID_COL: str})
    pop = u[POP_COL].to_numpy(float)
    dist = np.load(IN_DIR / DIST_FILE.format(**fmt), mmap_mode="r")
    n_c = float(np.load(IN_DIR / OBS_FILE.format(**fmt), mmap_mode="r").sum(dtype=np.float64))
    return pop, dist, n_c, pop * (n_c / pop.sum())


rows = []
for country in RUN_COUNTRIES:
    pop, dist, n_c, out_trips = load_inputs(country, LEVELS[country])
    n = len(pop)
    path = OUT_DIR / MAT_FILE.format(country=country)
    t0 = time.time()
    if path.exists():
        mat = np.load(path)
        status = "loaded"
    else:
        keep = pop > 0                                          # zero-population units are left out and re-inserted as zero rows/columns
        dist_k = np.asarray(dist[np.ix_(keep, keep)], dtype=np.float64)
        try:
            sim = tdlm.run_law_model(
                law=LAW, mass_origin=pop[keep], mass_destination=pop[keep], distance=dist_k,
                exponent=np.array([RAD_EXPONENT]), model="UM", out_trips=out_trips[keep],
                average=True, verbose=False)
        except Exception as exc:
            # print only the exception class name (no paths in the output)
            print(country, "| failed:", type(exc).__name__)
            continue
        mat_k = np.asarray(sim if not isinstance(sim, dict) else next(iter(sim.values())))[0]
        mat = np.zeros((n, n), dtype=np.float32)
        mat[np.ix_(keep, keep)] = mat_k
        np.save(path, mat)                                       # checkpoint per country
        status = "computed"
        del sim, mat_k, dist_k
    seconds = round(time.time() - t0, 1)

    # checks
    if np.isnan(mat).any() or (mat < 0).any():
        raise ValueError(f"{country}: a mátrixban hiányzó vagy negatív érték van")
    total, diag = float(mat.sum(dtype=np.float64)), float(np.trace(mat))
    row_sum = mat.sum(axis=1, dtype=np.float64)
    row_err = float(np.max(np.abs(row_sum - out_trips) / np.where(out_trips > 0, out_trips, 1.0)))
    peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 ** 2
    print(f"[{country}] {status} | összeg {total:,.1f} (N_c {n_c:,.0f}) | átló {diag} | "
          f"sorösszeg legnagyobb relatív eltérése az out_trips-től: {row_err:.2e} | {seconds} s | csúcsmemória {peak_gb:.2f} GB")
    if abs(total / n_c - 1) > SUM_RTOL_RUN:
        raise ValueError(f"{country}: a mátrix összege nem egyezik az N_c értékkel")
    if row_err > ROW_RTOL:
        raise ValueError(f"{country}: a sorösszegek nem egyeznek az out_trips értékkel")
    rows.append(dict(country=country, law=LAW, status=status, n_units=n, n_zero_pop=int((pop == 0).sum()),
                     exponent=RAD_EXPONENT, N_c=n_c, matrix_sum=total, diagonal_sum=diag,
                     max_flow=float(mat.max()), row_sum_max_rel_err=row_err, seconds=seconds, peak_memory_gb=round(peak_gb, 2)))
    del mat, pop, dist

# merge into the summary table (rows of the countries run now are replaced)
new = pd.DataFrame(rows)
if SUMMARY_CSV.exists() and len(new):
    old = pd.read_csv(SUMMARY_CSV)
    new = pd.concat([old[~old["country"].isin(new["country"])], new], ignore_index=True)
if len(new):
    new.to_csv(SUMMARY_CSV, index=False)
    print(f"\nMentve: {SUMMARY_CSV.as_posix()}")
    print(new.round(4).to_string(index=False))

In [ ]:
# --- Config (all changes are made here) ---
RUN_COUNTRIES = ["CZ"]            # the constants and load_inputs() come from the previous cell

rows = []
for country in RUN_COUNTRIES:
    pop, dist, n_c, out_trips = load_inputs(country, LEVELS[country])
    n = len(pop)
    path = OUT_DIR / MAT_FILE.format(country=country)
    t0 = time.time()
    if path.exists():
        mat = np.load(path)
        status = "loaded"
    else:
        keep = pop > 0                                          # zero-population units are left out and re-inserted as zero rows/columns
        dist_k = np.asarray(dist[np.ix_(keep, keep)], dtype=np.float64)
        try:
            sim = tdlm.run_law_model(
                law=LAW, mass_origin=pop[keep], mass_destination=pop[keep], distance=dist_k,
                exponent=np.array([RAD_EXPONENT]), model="UM", out_trips=out_trips[keep],
                average=True, verbose=False)
        except Exception as exc:
            # print only the exception class name (no paths in the output)
            print(country, "| failed:", type(exc).__name__)
            continue
        mat_k = np.asarray(sim if not isinstance(sim, dict) else next(iter(sim.values())))[0]
        mat = np.zeros((n, n), dtype=np.float32)
        mat[np.ix_(keep, keep)] = mat_k
        np.save(path, mat)                                       # checkpoint per country
        status = "computed"
        del sim, mat_k, dist_k
    seconds = round(time.time() - t0, 1)

    # checks
    if np.isnan(mat).any() or (mat < 0).any():
        raise ValueError(f"{country}: a mátrixban hiányzó vagy negatív érték van")
    total, diag = float(mat.sum(dtype=np.float64)), float(np.trace(mat))
    row_sum = mat.sum(axis=1, dtype=np.float64)
    row_err = float(np.max(np.abs(row_sum - out_trips) / np.where(out_trips > 0, out_trips, 1.0)))
    peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 ** 2
    print(f"[{country}] {status} | összeg {total:,.1f} (N_c {n_c:,.0f}) | átló {diag} | "
          f"sorösszeg legnagyobb relatív eltérése az out_trips-től: {row_err:.2e} | {seconds} s | csúcsmemória {peak_gb:.2f} GB")
    if abs(total / n_c - 1) > SUM_RTOL_RUN:
        raise ValueError(f"{country}: a mátrix összege nem egyezik az N_c értékkel")
    if row_err > ROW_RTOL:
        raise ValueError(f"{country}: a sorösszegek nem egyeznek az out_trips értékkel")
    rows.append(dict(country=country, law=LAW, status=status, n_units=n, n_zero_pop=int((pop == 0).sum()),
                     exponent=RAD_EXPONENT, N_c=n_c, matrix_sum=total, diagonal_sum=diag,
                     max_flow=float(mat.max()), row_sum_max_rel_err=row_err, seconds=seconds, peak_memory_gb=round(peak_gb, 2)))
    del mat, pop, dist

# merge into the summary table (rows of the countries run now are replaced)
new = pd.DataFrame(rows)
if SUMMARY_CSV.exists() and len(new):
    old = pd.read_csv(SUMMARY_CSV)
    new = pd.concat([old[~old["country"].isin(new["country"])], new], ignore_index=True)
if len(new):
    new.to_csv(SUMMARY_CSV, index=False)
    print(f"\nMentve: {SUMMARY_CSV.as_posix()}")
    print(new.round(4).to_string(index=False))